In [1]:
import pandas as pd
import numpy as np

In [2]:
train_data = pd.read_csv("./data/train.csv")
test_data = pd.read_csv("./data/test.csv")

train_data.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


# 1. Создаем Baseline модель

### Проводим небольшой EDA для baseline модели

In [3]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


Пропуски обнаружены в столбцах Cabin, Age, Embarked. Заполним медианой

In [4]:
train_data["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Сильного дисбаланса классов не обнаружено

In [5]:
train_data["Sex"].value_counts(normalize=True)

Sex
male      0.647587
female    0.352413
Name: proportion, dtype: float64

In [6]:
train_data.groupby("Sex")["Survived"].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [7]:
train_data.groupby("Pclass")["Survived"].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Классы Sex и Pclass сильно влияют на выживаемость. Будем использовать их в baseline модели

### Создаем выборки под baseline модели

In [8]:
baseline_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X_baseline = train_data[baseline_features].copy()
y = train_data["Survived"].copy()

In [9]:
from sklearn.model_selection import train_test_split

X_train_base, X_valid_base, y_train, y_valid = train_test_split(X_baseline, y, test_size=0.2, random_state=42,stratify=y)

#### Заполняем пропуски значений

In [10]:
X_train_base_processed = X_train_base.copy()
X_valid_base_processed = X_valid_base.copy()

In [20]:
age_median = X_train_base_processed["Age"].median()

X_train_base_processed["Age"] = X_train_base_processed["Age"].fillna(age_median)
X_valid_base_processed["Age"] = X_valid_base_processed["Age"].fillna(age_median)

embarked_moda = X_train_base_processed["Embarked"].mode()[0]

X_train_base_processed["Embarked"] = X_train_base_processed["Embarked"].fillna(embarked_moda)
X_valid_base_processed["Embarked"] = X_valid_base_processed["Embarked"].fillna(embarked_moda)


#### Обрабатываем категориальные признаки

In [23]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Sex", "Embarked"]

ohe = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)

ohe.fit(X_train_base_processed[categorical_features])

X_train_cat = ohe.transform(X_train_base_processed[categorical_features])
X_valid_cat = ohe.transform(X_valid_base_processed[categorical_features])

In [28]:
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

ohe_feature_names = ohe.get_feature_names_out(categorical_features)

X_train_cat_df = pd.DataFrame(
    X_train_cat,
    columns=ohe_feature_names,
    index=X_train_base_processed.index
)

X_valid_cat_df = pd.DataFrame(
    X_valid_cat,
    columns=ohe_feature_names,
    index=X_valid_base_processed.index
)

X_train_base_processed = pd.concat(
    [
        X_train_base_processed[numeric_features],
        X_train_cat_df
    ],
    axis=1
)

X_valid_base_processed = pd.concat(
    [
        X_valid_base_processed[numeric_features],
        X_valid_cat_df
    ],
    axis=1
)


#### Обучаем модель логистической регрессии и считаем метрику Accuracy

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_reg = LogisticRegression()
log_reg.fit(X_train_base_processed, y_train)
y_pred_log = log_reg.predict(X_valid_base_processed)

accuracy = accuracy_score(y_valid, y_pred_log)
print(accuracy)


0.7988826815642458


c:\Users\user\Desktop\IT\NLP\Pytorch основы\stepik\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Первая baseline-модель Logistic Regression показала accuracy ≈ 0.799 на validation. Это заметно выше наивного baseline ≈ 0.616, значит модель уже улавливает полезные закономерности в данных. Для первой простой модели результат можно считать хорошим.